In [1]:
import os
print(os.getcwd())

/Users/jwitter24/Desktop/Final Project


In [2]:
import os
print(os.path.expanduser('~'))

/Users/jwitter24


In [3]:
import pandas as pd
import re

INPUT_FILE = '/Users/jwitter24/Desktop/Final Project/my_dataset.csv'
OUTPUT_FILE = '/Users/jwitter24/Desktop/Final Project/ward3_addresses.csv'

#I gleaned all these stret names from a DC source with a map of Ward 3 with all streets in it. The text was easily extracted from there. I then expanded the abbreviations to match the format of the addresses in my dataset. I also sorted them by length to ensure that longer street names are matched before shorter ones (to avoid partial matches).
ward3_streets = [
    "17TH ST","18TH ST","19TH ST","20TH ST","21ST ST","22ND ST","23RD ST","24TH ST","25TH ST",
    "26TH ST","27TH ST","28TH PL","28TH ST","29TH PL","29TH ST","30TH PL","30TH ST","31ST PL",
    "31ST ST","32ND PL","32ND ST","33RD PL","33RD ST","34TH PL","34TH ST","35TH PL","35TH ST",
    "36TH PL","36TH ST","37TH ST","38TH ST","39TH PL","39TH ST","40TH PL","40TH ST","41ST PL",
    "41ST ST","42ND PL","42ND ST","43RD PL","43RD ST","44TH PL","44TH ST","45TH ST","46TH ST",
    "47TH PL","47TH ST","48TH PL","48TH ST","49TH ST","50TH PL","50TH ST","51ST PL","51ST ST",
    "52ND CT","52ND ST","52ND TER","ABERFOYLE PL","ADAMS MILL RD","ALBEMARLE ST","ALLAN RD",
    "ALLEN PL","ALLENDALE PL","ALLISON ST","ALTON PL","APPLETON ST","ARCADIA PL","ARGYLE TER",
    "ARIZONA AVE","ARIZONA TER","ASBURY PL","ASHBY ST","ASHMEAD PL","AUDUBON TER","AVON LN",
    "AVON PL","BANCROFT PL","BARNABY ST","BATTERY PL","BEACH DR","BEECH ST","BEECHER ST",
    "BELMONT RD","BELT RD","BENDING LN","BENTON PL","BENTON ST","BILTMORE ST","BINGHAM DR",
    "BLAGDEN AVE","BLAGDEN TER","BRANDYWINE ST","BROAD BRANCH RD","BURLINGTON PL",
    "BUTTERWORTH PL","C ST","CALIFORNIA ST","CALVERT ST","CANAL RD","CAROLINA PL",
    "CATHEDRAL AVE","CATON PL","CECIL PL","CHAIN BRIDGE RD","CHAMPLAIN ST","CHESAPEAKE ST",
    "CHESTERFIELD PL","CHESTNUT LN","CHEVY CHASE PKWY","CHURCH ST","CLARA BARTON PKWY",
    "CLARK PL","CLEVELAND AVE","CLYDESDALE PL","COLORADO AVE","COLUMBIA RD","CONNECTICUT AVE",
    "CONSTITUTION AVE","COREY PL","CORTLAND PL","CRESTWOOD DR","CUMBERLAND ST","CUSHING PL",
    "D ST","DALECARLIA PKWY","DALECARLIA PL","DANA PL","DAVENPORT ST","DAVIS PL","DAVIS RD",
    "DAVIS ST","DE SALES ST","DECATUR PL","DENT PL","DEVONSHIRE PL","DEXTER ST","DEXTER TER",
    "DONALDSON PL","DORSETT PL","DUMBARTON ST","DUNMORE LN","E ST","EAST PL","EDGEVALE TER",
    "EDMUNDS PL","EDMUNDS ST","ELLICOTT ST","ELLICOTT TER","ELLIOT PL","EMERY PL","ESKRIDGE TER",
    "EUCLID ST","EVERETT ST","F ST","FARADAY PL","FESSENDEN ST","FLORIDA AVE","FORDHAM RD",
    "FOREST LN","FORT DR","FOXHALL CRES","FOXHALL RD","FOXVIEW CIR","FULTON ST","G ST",
    "GALENA PL","GARFIELD ST","GARFIELD TER","GARRISON ST","GATES RD","GILLIS AVE",
    "GLENBROOK RD","GLENBROOK TER","GLOVER DR","GLOVER RD","GRACE ST","GRAMERCY ST","GRANT RD",
    "GREENE PL","H ST","HADFIELD LN","HALL PL","HARRISON ST","HARVARD ST","HAWTHORNE PL",
    "HAWTHORNE ST","HIGHLAND PL","HIGHWOOD CT","HILL RD","HILLANDALE DR","HILLBROOK LN",
    "HILLYER PL","HOBAN RD","HOBART ST","HOPKINS ST","HORSE STABLE RD","HOWARD ST",
    "HUNTINGTON ST","HURST TER","HUTCHINS PL","I ST","IDAHO AVE","INGLESIDE TER","INGOMAR PL",
    "INGOMAR ST","INTERNATIONAL DR","IRVING ST","JEFFERSON PL","JENIFER ST","JOCELYN ST",
    "JOYCE RD","K ST","KALORAMA CIR","KALORAMA RD","KANAWHA ST","KENMORE DR","KENYON ST",
    "KILBOURNE PL","KING PL","KLINGLE PL","KLINGLE RD","KLINGLE ST","L ST","LAMONT ST",
    "LANIER PL","LAVEROCK PL","LEGATION ST","LENORE LN","LEROY PL","LIBRARY WALK","LINGAN RD",
    "LINNEAN AVE","LINNEAN TER","LITTLE FALLS RD","LIVINGSTON ST","LOUGHBORO RD","LOVERS LN",
    "LOWELL LN","LOWELL ST","M ST","MACARTHUR BLVD","MACARTHUR TER","MACOMB ST","MANNING PL",
    "MANOR PL","MANSION CT","MANSION DR","MASSACHUSETTS AVE","MATHEWSON DR","MAUD ST","MAURY PL",
    "MCGILL TER","MCKINLEY PL","MCKINLEY ST","MILITARY RD","MILL RD","MILLWOOD LN","MINTWOOD PL",
    "MONROE ST","MORELAND PL","MORELAND ST","MORGAN LN","MORRISON ST","MORROW DR","N ST",
    "NEBRASKA AVE","NEVADA AVE","NEW MEXICO AVE","NEW YORK AVE","NEWARK ST","NEWLANDS ST",
    "NEWPORT PL","NEWTON ST","NORMANSTONE DR","NORMANSTONE TER","NORTH RD","NORTHAMPTON ST",
    "NORTON PL","O ST","OBSERVATORY CIR","OBSERVATORY LN","OHIO DR","OLIVE ST","OLIVER ST",
    "ONTARIO PL","ONTARIO RD","ORDWAY ST","OREGON AVE","OVERLOOK RD","P ST","PALISADE LN",
    "PARK AVE","PARK RD","PARKGLEN CT","PARTRIDGE LN","PATTERSON PL","PATTERSON ST",
    "PENNSYLVANIA AVE","PHELPS PL","PINEY BRANCH PKWY","POPLAR ST","PORTER ST","POTOMAC AVE",
    "POTOMAC ST","PROSPECT ST","Q LN","Q PL","Q ST","QUEBEC PL","QUEBEC ST","QUESADA ST",
    "QUINCY ST","R ST","RANDOLPH ST","RENO RD","RESERVOIR RD","RIDGE RD","RIGGS PL",
    "RITTENHOUSE ST","RIVER RD","ROCK CREEK DR","ROCKWOOD PKWY","RODMAN ST","ROSEMOUNT AVE",
    "ROSS DR","ROSS PL","ROWLAND PL","RUNNYMEDE PL","S ST","SALEM LN","SEATON ST","SEDGWICK ST",
    "SHEPHERD ST","SHERIER PL","SHOEMAKER ST","SHOREHAM DR","SOUTH ST","SPRINGDALE ST",
    "STEPHENSON LN","STEPHENSON PL","STUYVESANT PL","SUMMIT PL","SUNDERLAND PL","SURREY LN",
    "SUTERS LN","SWANN ST","T ST","TAYLOR ST","TENLEY CIR","TENNYSON ST","TILDEN PL","TILDEN ST",
    "TONDORF RD","TRACY PL","TRUMBULL TER","TUNLAW RD","U ST","UNICORN LN","UNIVERSITY AVE",
    "UNIVERSITY TER","UPLAND TER","UPSHUR ST","UPTON ST","UPTON TER","UTAH AVE","V ST",
    "VAN NESS ST","VARNUM ST","VEAZEY ST","VEAZEY TER","VERNON ST","VIRGINIA AVE","VOLTA PL",
    "W ST","WALBRIDGE PL","WARD PL","WARREN PL","WARREN ST","WATER ST","WATERSIDE DR",
    "WATSON PL","WATSON ST","WEAVER TER","WESLEY CIR","WEST RD","WESTERN AVE","WHITEHAVEN PKWY",
    "WHITEHAVEN ST","WILLARD ST","WILLIAMSBURG LN","WINDOM PL","WINFIELD LN","WISCONSIN AVE",
    "WOODLAND DR","WOODLEY PL","WOODLEY RD","WOODWAY LN","WORTHINGTON ST","WYOMING AVE",
    "YATES RD","YUMA CT","YUMA PL","YUMA ST",
]

#This expands the street suffixes to their full forms to ensure that they match the format of the addresses in the dataset. For example, "ST" becomes "STREET", "AVE" becomes "AVENUE", etc. This is important because the addresses in the dataset may use the full forms of the street names, and we want to ensure that our regex pattern matches them correctly.
suffix_map = {
    'ST': 'STREET', 'AVE': 'AVENUE', 'PL': 'PLACE', 'RD': 'ROAD',
    'TER': 'TERRACE', 'CT': 'COURT', 'CIR': 'CIRCLE', 'DR': 'DRIVE',
    'LN': 'LANE', 'PKWY': 'PARKWAY', 'WALK': 'WALK', 'WAY': 'WAY',
    'BLVD': 'BOULEVARD',
}

def expand_suffix(name):
    parts = name.split()
    if parts[-1] in suffix_map:
        parts[-1] = suffix_map[parts[-1]]
    return ' '.join(parts)

expanded = sorted(set(expand_suffix(s) for s in ward3_streets), key=len, reverse=True)

#This creates a regex pattern that matches any of the expanded street names as whole words. The re.escape is used to escape any special characters in the street names, ensuring that they are treated as literal strings in the regex pattern.
pattern = re.compile(r'\b(?:' + '|'.join(re.escape(s) for s in expanded) + r')\b')

# --- LOAD & FILTER ---
print("Loading dataset...")
df = pd.read_csv(INPUT_FILE, low_memory=False)
print(f"  Total rows: {len(df):,}")

# Filtering the NW addresses on Ward 3 streets. 
ward3_mask = (
    df['FULLADDRESS'].str.contains(' NW', na=False) &
    df['FULLADDRESS'].str.contains(pattern, na=False)
)

filtered = df[ward3_mask].copy()
print(f"  Ward 3 matches: {len(filtered):,}")

filtered.to_csv(OUTPUT_FILE, index=False)

Loading dataset...
  Total rows: 126,809
  Ward 3 matches: 35,663


In [4]:
#Due to the limitations of the geocoding API, I had to split the geocoding process into chunks. Each chunk was geocoded separately and saved as an Excel file. Now, I need to load all those geocoded files, combine them into a single DataFrame, and then save the final combined results as a CSV file. Additionally, I want to separate the successfully geocoded addresses from those that didn't match (i.e., those with NaN in the 'MAR_SCORE' column) and save them as separate CSV files for further analysis or reprocessing.

import pandas as pd
import glob
import os

# Path to the directory containing the geocoded chunk files
results_dir = '/Users/jwitter24/Desktop/Final Project/mar_chunks_geocoded'

#Load and combine all geocoded files
files = sorted(glob.glob(f'{results_dir}/ward3_chunk_*_Geocoded.xlsx'))
print(f"Found {len(files)} geocoded files...")

dfs = []
for f in files:
    df = pd.read_excel(f)
    dfs.append(df)
    print(f"  {os.path.basename(f)}: {len(df)} rows ({df['MAR_SCORE'].notna().sum()} matched)")

#Combine
combined = pd.concat(dfs, ignore_index=True)
print(f"\nTotal rows: {len(combined):,}")
print(f"Successfully geocoded: {combined['MAR_SCORE'].notna().sum():,}")
print(f"Unmatched: {combined['MAR_SCORE'].isna().sum():,}")

#Spliting into matched and unmatched
matched = combined[combined['MAR_SCORE'].notna()].copy()
unmatched = combined[combined['MAR_SCORE'].isna()][['ADDRESS']].copy()

# Save
matched.to_csv('/Users/jwitter24/Desktop/Final Project/ward3_geocoded_final.csv', index=False)
unmatched.to_csv('/Users/jwitter24/Desktop/Final Project/ward3_unmatched.csv', index=False)

print(f"  ward3_geocoded_final.csv  → {len(matched):,} geocoded addresses")
print(f"  ward3_unmatched.csv       → {len(unmatched):,} addresses that didn't match")

Found 36 geocoded files...
  ward3_chunk_01_Geocoded.xlsx: 1000 rows (999 matched)
  ward3_chunk_02_Geocoded.xlsx: 1000 rows (996 matched)
  ward3_chunk_03_Geocoded.xlsx: 1000 rows (997 matched)
  ward3_chunk_04_Geocoded.xlsx: 1000 rows (994 matched)
  ward3_chunk_05_Geocoded.xlsx: 1000 rows (999 matched)
  ward3_chunk_06_Geocoded.xlsx: 1000 rows (993 matched)
  ward3_chunk_07_Geocoded.xlsx: 1000 rows (999 matched)
  ward3_chunk_08_Geocoded.xlsx: 1000 rows (995 matched)
  ward3_chunk_09_Geocoded.xlsx: 1000 rows (999 matched)
  ward3_chunk_10_Geocoded.xlsx: 1000 rows (996 matched)
  ward3_chunk_11_Geocoded.xlsx: 1000 rows (999 matched)
  ward3_chunk_12_Geocoded.xlsx: 1000 rows (996 matched)
  ward3_chunk_13_Geocoded.xlsx: 1000 rows (986 matched)
  ward3_chunk_14_Geocoded.xlsx: 1000 rows (996 matched)
  ward3_chunk_15_Geocoded.xlsx: 1000 rows (996 matched)
  ward3_chunk_16_Geocoded.xlsx: 1000 rows (995 matched)
  ward3_chunk_17_Geocoded.xlsx: 1000 rows (991 matched)
  ward3_chunk_18_Geoc

In [5]:
import pandas as pd
import glob
import os

results_dir = '/Users/jwitter24/Desktop/Final Project/mar_chunks_geocoded'

files = sorted(glob.glob(f'{results_dir}/ward3_chunk_*_Geocoded.xlsx'))
print(f"Found {len(files)} geocoded files...")

dfs = []
for f in files:
    df = pd.read_excel(f)
    dfs.append(df)
    print(f"  {os.path.basename(f)}: {len(df)} rows ({df['MAR_SCORE'].notna().sum()} matched)")

combined = pd.concat(dfs, ignore_index=True)
print(f"\nTotal rows: {len(combined):,}")
print(f"Successfully geocoded: {combined['MAR_SCORE'].notna().sum():,}")
print(f"Unmatched: {combined['MAR_SCORE'].isna().sum():,}")

matched = combined[combined['MAR_SCORE'].notna()].copy()
unmatched = combined[combined['MAR_SCORE'].isna()][['ADDRESS']].copy()

matched.to_csv('/Users/jwitter24/Desktop/Final Project/ward3_geocoded_final.csv', index=False)
unmatched.to_csv('/Users/jwitter24/Desktop/Final Project/ward3_unmatched.csv', index=False)

print(f"  ward3_geocoded_final.csv  → {len(matched):,} geocoded addresses")
print(f"  ward3_unmatched.csv       → {len(unmatched):,} addresses that didn't match")

Found 36 geocoded files...
  ward3_chunk_01_Geocoded.xlsx: 1000 rows (999 matched)
  ward3_chunk_02_Geocoded.xlsx: 1000 rows (996 matched)
  ward3_chunk_03_Geocoded.xlsx: 1000 rows (997 matched)
  ward3_chunk_04_Geocoded.xlsx: 1000 rows (994 matched)
  ward3_chunk_05_Geocoded.xlsx: 1000 rows (999 matched)
  ward3_chunk_06_Geocoded.xlsx: 1000 rows (993 matched)
  ward3_chunk_07_Geocoded.xlsx: 1000 rows (999 matched)
  ward3_chunk_08_Geocoded.xlsx: 1000 rows (995 matched)
  ward3_chunk_09_Geocoded.xlsx: 1000 rows (999 matched)
  ward3_chunk_10_Geocoded.xlsx: 1000 rows (996 matched)
  ward3_chunk_11_Geocoded.xlsx: 1000 rows (999 matched)
  ward3_chunk_12_Geocoded.xlsx: 1000 rows (996 matched)
  ward3_chunk_13_Geocoded.xlsx: 1000 rows (986 matched)
  ward3_chunk_14_Geocoded.xlsx: 1000 rows (996 matched)
  ward3_chunk_15_Geocoded.xlsx: 1000 rows (996 matched)
  ward3_chunk_16_Geocoded.xlsx: 1000 rows (995 matched)
  ward3_chunk_17_Geocoded.xlsx: 1000 rows (991 matched)
  ward3_chunk_18_Geoc

In [6]:
import pandas as pd

#Loading original ward3 dataset
ward3 = pd.read_csv('/Users/jwitter24/Desktop/Final Project/ward3_addresses.csv', low_memory=False)

#Loading the geocoded results
geocoded = pd.read_csv('/Users/jwitter24/Desktop/Final Project/ward3_geocoded_final.csv')

# Preview what columns MAR gave us
print("MAR columns:", geocoded.columns.tolist())
print(f"Ward3 rows: {len(ward3):,}")
print(f"Geocoded rows: {len(geocoded):,}")

MAR columns: ['ADDRESS', 'CITY', 'STATE', 'MAR_MARID', 'MAR_MATCHADDRESS', 'MAR_SCORE', 'MAR_SSL', 'MAR_ALIAS', 'MAR_XCOORD', 'MAR_YCOORD', 'MAR_LATITUDE', 'MAR_LONGITUDE', 'MAR_ADDRNUM', 'MAR_ADDRNUMSUFFIX', 'MAR_STNAME', 'MAR_STREETTYPE', 'MAR_QUADRANT', 'MAR_ZIPCODE', 'MAR_BLOCKKEY', 'MAR_SUBBLOCKKEY', 'MAR_WARD', 'MAR_ANC', 'MAR_CENSUSTRACT', 'MAR_RESIDENCETYPE', 'MAR_HASCONDOUNIT', 'MAR_HASRESUNIT', 'MAR_STATUS', 'MAR_NATIONALGRID', 'MAR_MESSAGE']
Ward3 rows: 35,663
Geocoded rows: 35,303


In [7]:
import pandas as pd

ward3 = pd.read_csv('/Users/jwitter24/Desktop/Final Project/ward3_addresses.csv', low_memory=False)
geocoded = pd.read_csv('/Users/jwitter24/Desktop/Final Project/ward3_geocoded_final.csv')

#Keeping only the useful MAR columns (drop admin/redundant ones)
mar_cols = [
    'ADDRESS', 'MAR_MARID', 'MAR_MATCHADDRESS', 'MAR_SCORE',
    'MAR_SSL', 'MAR_LATITUDE', 'MAR_LONGITUDE',
    'MAR_WARD', 'MAR_ANC', 'MAR_CENSUSTRACT', 'MAR_ZIPCODE',
    'MAR_QUADRANT', 'MAR_RESIDENCETYPE', 'MAR_STATUS'
]
geocoded_slim = geocoded[mar_cols].copy()

#Join on address finnally, ward3 has FULLADDRESS, geocoded has ADDRESS (same values)
merged = ward3.merge(geocoded_slim, left_on='FULLADDRESS', right_on='ADDRESS', how='left')

#Drop the redundant ADDRESS column from MAR
merged.drop(columns=['ADDRESS'], inplace=True)

print(f"Original rows:  {len(ward3):,}")
print(f"Merged rows:    {len(merged):,}")
print(f"With coords:    {merged['MAR_LATITUDE'].notna().sum():,}")
print(f"Without coords: {merged['MAR_LATITUDE'].isna().sum():,}")

merged.to_csv('/Users/jwitter24/Desktop/Final Project/ward3_final.csv', index=False)

Original rows:  35,663
Merged rows:    35,663
With coords:    35,505
Without coords: 158


In [8]:
import pandas as pd

#Loading master geocoded file
ward3 = pd.read_csv('/Users/jwitter24/Desktop/Final Project/ward3_final.csv', low_memory=False)

# Loading original dataset.
original = pd.read_csv('/Users/jwitter24/Desktop/Final Project/my_dataset.csv', low_memory=False)

print("ward3 columns:", ward3.columns.tolist())
print("original columns:", original.columns.tolist())
print()
print(f"ward3 rows: {len(ward3):,}")
print(f"original rows: {len(original):,}")
print()
# Check if kit house columns are already in ward3, they are not, so I have to redo this process from earlier in the project where I add the kit house flags to the original dataset and then re-merge with the geocoded results.

ward3 columns: ['OBJECTID', 'FULLADDRESS', 'BuildingName', 'Square', 'Lot', 'Date', 'MapYear', 'Permit_Notes', 'NOTES', 'Suffix', 'EDITTYPE', 'House_Key', 'House_Type', 'Extant', 'Updated', 'Facade', 'Flag', 'Contributing', 'ArchitectGroup', 'Developer', 'Quantity', 'Rooms', 'Width', 'Depth', 'Number_of_Families', 'Store', 'Stories', 'Permits_Last_Editor', 'Permits_Last_Edit_Date', 'RazedFlag', 'PermitsSource', 'PermitNumber', 'Owner', 'Architect', 'Builder', 'Material', 'Purpose', 'EstimatedCost', 'MicrofilmRoll', 'PermitType', 'SolidFilled', 'FoundationMaterial', 'StoneType', 'RoofType', 'FrontMaterial', 'RoofMaterial', 'Heat', 'Shape__Area', 'Shape__Length', 'MAR_MARID', 'MAR_MATCHADDRESS', 'MAR_SCORE', 'MAR_SSL', 'MAR_LATITUDE', 'MAR_LONGITUDE', 'MAR_WARD', 'MAR_ANC', 'MAR_CENSUSTRACT', 'MAR_ZIPCODE', 'MAR_QUADRANT', 'MAR_RESIDENCETYPE', 'MAR_STATUS']
original columns: ['OBJECTID', 'FULLADDRESS', 'BuildingName', 'Square', 'Lot', 'Date', 'MapYear', 'Permit_Notes', 'NOTES', 'Suffix',

In [9]:
import pandas as pd
import re

#Loading your master geocoded ward3 file
ward3 = pd.read_csv('/Users/jwitter24/Desktop/Final Project/ward3_final.csv', low_memory=False)

#Flagging kit houses based on official architect field and hidden text fields (builder, permit notes, general notes)
kit_terms = ['Sears', 'Roebuck', 'Honor Bilt', 'Ready-cut', 'Kit House', 'Aladdin', 'Montgomery Ward']
kit_pattern = '|'.join(kit_terms)

#These are 100% official kits, where the architect field explicitly mentions a known kit company. This is the most reliable flag, but it will miss any kits that weren't properly documented in the architect field.
ward3['Official_Kit'] = ward3['Architect'].str.contains(kit_pattern, case=False, na=False)

#These are "hidden" kits that don't have the architect field filled out with a known kit company, but they do have clues in the builder, permit notes, or general notes fields. By creating a combined text blob of these fields and searching for kit-related terms, we can flag potential kit houses that might have been missed by the official architect flag. This is less reliable than the official flag, as it relies on the presence of certain keywords in unstructured text fields, but it can help identify additional candidates.
ward3['Text_Blob'] = ward3['Builder'].fillna('') + ' ' + ward3['Permit_Notes'].fillna('') + ' ' + ward3['NOTES'].fillna('')
ward3['Hidden_Kit'] = ward3['Text_Blob'].str.contains(kit_pattern, case=False, na=False)

#This creates a final flag that combines both the official architect-based kits and the hidden text-based kits. Any permit that is flagged as either an official kit or a hidden kit will be marked as a kit house in this final flag. This allows us to capture a broader set of potential kit houses, while still distinguishing between those that have strong evidence (official architect) and those that have weaker evidence (hidden text clues).
ward3['Is_Kit_House'] = ward3['Official_Kit'] | ward3['Hidden_Kit']


ward3.drop(columns=['Text_Blob'], inplace=True)

# Summary
print(f"Total Ward 3 permits:         {len(ward3):,}")
print(f"Official kits (Architect):    {ward3['Official_Kit'].sum()}")
print(f"Hidden kits (notes/builder):  {(ward3['Is_Kit_House'].sum() - ward3['Official_Kit'].sum())}")
print(f"Total kit houses found:       {ward3['Is_Kit_House'].sum()}")
print(f"With coordinates:             {ward3[ward3['Is_Kit_House'] & ward3['MAR_LATITUDE'].notna()].shape[0]}")

# Save
ward3.to_csv('/Users/jwitter24/Desktop/Final Project/ward3_final.csv', index=False)
print("\nSaved: ward3_final.csv — ready for mapping")

Total Ward 3 permits:         35,663
Official kits (Architect):    102
Hidden kits (notes/builder):  17
Total kit houses found:       119
With coordinates:             119

Saved: ward3_final.csv — ready for mapping


In [10]:
import pandas as pd
import numpy as np
import folium

#Loading final dataset
df = pd.read_csv('/Users/jwitter24/Desktop/Final Project/ward3_final.csv', low_memory=False)

# Filter to kit houses with coordinates
kits = df[df['Is_Kit_House'] & df['MAR_LATITUDE'].notna()].copy()
print(f"Mapping {len(kits)} kit houses...")

#Centeering map on ward 3.
m = folium.Map(location=[38.93, -77.07], zoom_start=13, tiles='CartoDB positron')

np.random.seed(42)
jitter = 0.0008

# I thought maybe houses would appear on top of each other if they have the same coordinates, so I added a small random jitter to the latitude and longitude of each point to help spread them out visually on the map. This proved to be uneccessary in the end, as there were very few exact coordinate duplicates and I removed it later.

for _, row in kits.iterrows():
    folium.CircleMarker(
        location=[
            row['MAR_LATITUDE'] + np.random.uniform(-jitter, jitter),
            row['MAR_LONGITUDE'] + np.random.uniform(-jitter, jitter)
        ],
        radius=5,
        popup=f"Address: {row['FULLADDRESS']}<br>Year: {row['MapYear']}",
        color='red',
        fill=True,
        fill_opacity=0.7
    ).add_to(m)

m

Mapping 119 kit houses...


In [11]:
import pandas as pd
import numpy as np
import folium

df = pd.read_csv('/Users/jwitter24/Desktop/Final Project/ward3_final.csv', low_memory=False)
kits = df[df['Is_Kit_House'] & df['MAR_LATITUDE'].notna()].copy()
print(f"Mapping {len(kits)} kit houses...")

def decade_color(year):
    if year < 1920: return 'blue'
    elif year < 1930: return 'red'
    elif year < 1940: return 'orange'
    else: return 'gray'

#This is pretty much what I did on my MVP but with the proper coordinates instead of random points, and with the addition of color-coding the points by decade and adding a legend to explain the colors. The popup for each point now also includes the builder information if available.

m = folium.Map(location=[38.93, -77.07], zoom_start=13, tiles='CartoDB positron')

for _, row in kits.iterrows():
    color = decade_color(row['MapYear'])
    folium.CircleMarker(
        location=[row['MAR_LATITUDE'], row['MAR_LONGITUDE']],
        radius=5,
        popup=f"Address: {row['FULLADDRESS']}<br>Year: {row['MapYear']}<br>Builder: {row['Builder']}",
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.8
    ).add_to(m)

legend_html = """
<div style="position: fixed; bottom: 40px; left: 40px; z-index: 1000;
     background-color: white; padding: 12px; border-radius: 8px;
     border: 1px solid #ccc; font-family: Arial; font-size: 13px;">
  <b>Kit House Era</b><br>
  <span style="color:blue">●</span> Pre-1920s<br>
  <span style="color:red">●</span> 1920s<br>
  <span style="color:orange">●</span> 1930s<br>
  <span style="color:gray">●</span> 1940s+
</div>
"""
m.get_root().html.add_child(folium.Element(legend_html))

m

Mapping 119 kit houses...


In [12]:
import pandas as pd
import numpy as np
import folium
import re

#Loading final dataset
df = pd.read_csv('/Users/jwitter24/Desktop/Final Project/ward3_final.csv', low_memory=False)
kits = df[df['Is_Kit_House'] & df['MAR_LATITUDE'].notna()].copy()
print(f"Mapping {len(kits)} kit houses...")

#Same stuff as before, but I also added a function to convert DMS (Degrees, Minutes, Seconds) coordinates to decimal degrees, in case I needed to convert any coordinates that were in DMS format. This function uses regular expressions to parse the DMS string and calculate the decimal degree values for latitude and longitude.
def dms_to_dd(dms_str):
    dms_str = dms_str.strip()
    pattern = r'(\d+)°(\d+)\'([\d.]+)\"([NS])\s+(\d+)°(\d+)\'([\d.]+)\"([EW])'
    m = re.match(pattern, dms_str)
    if not m:
        return None, None
    lat = int(m[1]) + int(m[2])/60 + float(m[3])/3600
    if m[4] == 'S': lat = -lat
    lon = int(m[5]) + int(m[6])/60 + float(m[7])/3600
    if m[8] == 'W': lon = -lon
    return lat, lon

#Color coding.
def decade_color(year):
    if year < 1920: return 'blue'
    elif year < 1930: return 'red'
    elif year < 1940: return 'orange'
    else: return 'gray'

#Defining the map.
m = folium.Map(location=[38.93, -77.07], zoom_start=13, tiles='CartoDB positron')

#Looked at historical maps of DC's streetcar lines and found the coordinates for the Glen Echo line, which ran through Ward 3 and likely influenced development patterns in the area. By adding this line to the map, we can provide historical context for why certain areas of Ward 3 may have more kit houses or different development patterns based on their proximity to the streetcar line. This adds an extra layer of insight to the map and helps tell a more complete story about the history of kit houses in Ward 3.
glen_echo_line = [
    [38.938583, -77.113722],
    [38.935667, -77.112806],
    [38.931889, -77.110056],
    [38.927056, -77.105333],
    [38.922000, -77.102611],
    [38.919306, -77.100333],
    [38.909639, -77.092333],
    [38.906222, -77.084639],
    [38.905917, -77.071472],
]

folium.PolyLine(
    glen_echo_line,
    color='#2196F3',
    weight=4,
    opacity=0.7,
    tooltip='Glen Echo Streetcar Line'
).add_to(m)


for _, row in kits.iterrows():
    color = decade_color(row['MapYear'])
    folium.CircleMarker(
        location=[row['MAR_LATITUDE'], row['MAR_LONGITUDE']],
        radius=5,
        popup=f"Address: {row['FULLADDRESS']}<br>Year: {row['MapYear']}<br>Builder: {row['Builder']}",
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.8
    ).add_to(m)

# Creating a custom legend to explain the color coding of the kit houses by decade and to indicate the location of the Glen Echo streetcar line. This legend is added as a fixed element on the map, so it remains visible as users interact with the map. The legend uses simple HTML and inline CSS for styling, making it easy to customize and ensure it fits well with the overall design of the map. This is a proof of concept for expanding streetcar analysis.
legend_html = """
<div style="position: fixed; bottom: 40px; left: 40px; z-index: 1000;
     background-color: white; padding: 12px; border-radius: 8px;
     border: 1px solid #ccc; font-family: Arial; font-size: 13px;">
  <b>Kit House Era</b><br>
  <span style="color:blue">●</span> Pre-1920s<br>
  <span style="color:red">●</span> 1920s<br>
  <span style="color:orange">●</span> 1930s<br>
  <span style="color:gray">●</span> 1940s+<br><br>
  <b>Streetcar Lines</b><br>
  <span style="color:#2196F3">━</span> Glen Echo Line
</div>
"""
m.get_root().html.add_child(folium.Element(legend_html))

m

Mapping 119 kit houses...


In [13]:
import pandas as pd
import numpy as np
import folium

df = pd.read_csv('/Users/jwitter24/Desktop/Final Project/ward3_final.csv', low_memory=False)
kits = df[df['Is_Kit_House'] & df['MAR_LATITUDE'].notna()].copy()
print(f"Mapping {len(kits)} kit houses...")

def decade_color(year):
    if year < 1920: return 'blue'
    elif year < 1930: return 'red'
    elif year < 1940: return 'orange'
    else: return 'gray'

m = folium.Map(location=[38.93, -77.07], zoom_start=13, tiles='CartoDB positron')

#All streetcar lines in the area circa 1920s, based on historical maps. I found the coordinates for these lines and added them to the map to provide additional historical context for the development patterns in Ward 3. By showing the locations of these streetcar lines, we can help explain why certain areas may have more kit houses or different development patterns based on their proximity to public transit at the time. This adds depth to the analysis and helps tell a more complete story about the history of kit houses in Ward 3.
glen_echo_line = [
    [38.938583, -77.113722],[38.935667, -77.112806],[38.931889, -77.110056],
    [38.927056, -77.105333],[38.922000, -77.102611],[38.919306, -77.100333],
    [38.909639, -77.092333],[38.906222, -77.084639],[38.905917, -77.071472],
    [38.906056, -77.062861],
]

wisconsin_line = [
    [38.906056, -77.062861],[38.915694, -77.067722],[38.922917, -77.073167],
    [38.928667, -77.073111],[38.934111, -77.072361],[38.948833, -77.080111],
    [38.951083, -77.080722],[38.960778, -77.085750],
]

river_road_line = [
    [38.948833, -77.080111],[38.949028, -77.080667],[38.956472, -77.091250],
]

connecticut_ave_line = [
    [38.923556, -77.051444],[38.967556, -77.077111],
]

folium.PolyLine(glen_echo_line, color='#2196F3', weight=4, opacity=0.7, tooltip='Glen Echo Line').add_to(m)
folium.PolyLine(wisconsin_line, color='#9C27B0', weight=4, opacity=0.7, tooltip='Wisconsin Ave Line').add_to(m)
folium.PolyLine(river_road_line, color='#FF9800', weight=4, opacity=0.7, tooltip='River Road Line').add_to(m)
folium.PolyLine(connecticut_ave_line, color='#E91E63', weight=4, opacity=0.7, tooltip='Connecticut Ave Line').add_to(m)

for _, row in kits.iterrows():
    color = decade_color(row['MapYear'])
    folium.CircleMarker(
        location=[row['MAR_LATITUDE'], row['MAR_LONGITUDE']],
        radius=5,
        popup=f"Address: {row['FULLADDRESS']}<br>Year: {row['MapYear']}<br>Builder: {row['Builder']}",
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.8
    ).add_to(m)

#Legend
legend_html = """
<div style="position: fixed; bottom: 40px; left: 40px; z-index: 1000;
     background-color: white; padding: 12px; border-radius: 8px;
     border: 1px solid #ccc; font-family: Arial; font-size: 13px;">
  <b>Kit House Era</b><br>
  <span style="color:blue">●</span> Pre-1920s<br>
  <span style="color:red">●</span> 1920s<br>
  <span style="color:orange">●</span> 1930s<br>
  <span style="color:gray">●</span> 1940s+<br><br>
  <b>Streetcar Lines (c. 1920s)</b><br>
  <span style="color:#2196F3">━</span> Glen Echo Line<br>
  <span style="color:#9C27B0">━</span> Wisconsin Ave Line<br>
  <span style="color:#FF9800">━</span> River Road Line<br>
  <span style="color:#E91E63">━</span> Connecticut Ave Line
</div>
"""
m.get_root().html.add_child(folium.Element(legend_html))

m

Mapping 119 kit houses...


In [14]:
import pandas as pd

tax = pd.read_csv('/Users/jwitter24/Desktop/Final Project/dc_tax_assessment.csv', low_memory=False)
print(f"Rows: {len(tax):,}")
print(f"Columns: {tax.columns.tolist()}")
print()
print(tax.head(3).to_string())

#I want to add another layer of analysis to the map by incorporating tax assessment data. By merging the tax assessment data with our existing dataset of kit houses, we can analyze whether there are any correlations between being a kit house and having a certain level of tax assessment. This could provide insights into the economic status of kit houses compared to non-kit houses in Ward 3, and whether kit houses tend to be valued higher or lower than other properties in the area. This adds an extra dimension to our analysis and helps us understand the financial implications of owning a kit house in Ward 3.

Rows: 221,144
Columns: ['OBJECTID', 'SSL', 'SQUARE', 'SUFFIX', 'LOT', 'ARN', 'ASRNAME', 'PROPTYPE', 'TRIGROUP', 'USECODE', 'LANDAREA', 'PREMISEADD', 'NBHD', 'SUBNBHD', 'UNITNUMBER', 'OWNERNAME', 'CAREOFNAME', 'ADDRESS1', 'ADDRESS2', 'CITYSTZIP', 'OLDLAND', 'OLDIMPR', 'OLDTOTAL', 'NEWLAND', 'NEWIMPR', 'NEWTOTAL', 'PHASELAND', 'PHASEBUILD', 'PARTPART', 'VACLNDUSE', 'LOWNUMBER', 'STREETNAME', 'QDRNTNAME', 'DELCODE', 'HSTDCODE', 'CLASSTYPE', 'TAXRATE', 'MIXEDUSE', 'MIX1TXTYPE', 'MIX1CLASS', 'MIX1RATE', 'MIX1LNDPCT', 'MIX1LNDVAL', 'MIX1BLDPCT', 'MIX1BLDVAL', 'MIX2TXTYPE', 'MIX2CLASS', 'MIX2RATE', 'MIX2LNDPCT', 'MIX2LNDVAL', 'MIX2BLDPCT', 'MIX2BLDVAL', 'OWNOCCT', 'COOPUNITS', 'PCHILDCODE', 'ABTLOTCODE', 'SALEPRICE', 'SALEDATE', 'ACCEPTCODE', 'SALETYPE', 'DEEDDATE', 'ASSESSMENT', 'ANNUALTAX', 'DUEDATE1', 'AMTDUE1', 'DUEDATE2', 'AMTDUE2', 'DUEDATE3', 'AMTDUE3', 'TOTDUEAMT', 'TOTCOLAMT', 'TOTBALAMT', 'EXTRACTDAT', 'CAPCURR', 'CAPPROP', 'REASONCD', 'CY1YEAR', 'CY1TXSALE', 'CY1TAX', 'CY1PEN', 'CY

In [15]:
import pandas as pd

#Loading both files
ward3 = pd.read_csv('/Users/jwitter24/Desktop/Final Project/ward3_final.csv', low_memory=False)
tax = pd.read_csv('/Users/jwitter24/Desktop/Final Project/dc_tax_assessment.csv', low_memory=False)

# Keeping only the useful columns from tax data
tax_slim = tax[[
    'SSL', 'NEWTOTAL', 'NEWLAND', 'NEWIMPR',
    'SALEPRICE', 'SALEDATE', 'OWNERNAME', 'NBHDNAME', 'PROPTYPE'
]].copy()

# Cleaning up SSL format for joining - check what they look like in each file
print("ward3 MAR_SSL sample:  ", ward3['MAR_SSL'].dropna().head(5).tolist())
print("tax SSL sample:        ", tax_slim['SSL'].dropna().head(5).tolist())


ward3 MAR_SSL sample:   ['1301    0739', '1301    0735', '1301    0736', '1301    0738', '1301    0737']
tax SSL sample:         ['PI0004730345', 'PI0004730355', 'PI0004730344', 'PI0002570034', 'PI0S58680417']


In [16]:
# Check what PREMISEADD looks like vs FULLADDRESS
print("PREMISEADD samples:")
print(tax['PREMISEADD'].dropna().head(10).tolist())
print()
print("FULLADDRESS samples:")
print(ward3['FULLADDRESS'].dropna().head(10).tolist())

PREMISEADD samples:
['771 WHARF ST SW WASHINGTON DC 20024', '771 WHARF ST SW WASHINGTON DC 20024', '771 WHARF ST SW WASHINGTON DC 20024', '1300 E ST NW WASHINGTON DC 20004', '1100 OAK DRIVE SE WASHINGTON DC 20032', '1300 PENNSYLVANIA AVE NW WASHINGTON DC 20004', '1300 PENNSYLVANIA AVE NW WASHINGTON DC 20004', '900 WAHLER PL SE WASHINGTON DC 20032-4006', 'NW WASHINGTON DC 00000', '1100 OHIO DR SW WASHINGTON DC 20242']

FULLADDRESS samples:
['2216 38TH STREET NW', '2224 38TH STREET NW', '2222 38TH STREET NW', '2218 38TH STREET NW', '2220 38TH STREET NW', '3411 33RD PLACE NW', '1746 MASSACHUSETTS AVENUE NW', '1301 S STREET NW', '3554 APPLETON STREET NW', '3548 APPLETON STREET NW']


In [17]:
import pandas as pd
import re

ward3 = pd.read_csv('/Users/jwitter24/Desktop/Final Project/ward3_final.csv', low_memory=False)
tax = pd.read_csv('/Users/jwitter24/Desktop/Final Project/dc_tax_assessment.csv', low_memory=False)

# Strip city/state/zip from PREMISEADD and everything before " WASHINGTON DC"
tax['ADDRESS_CLEAN'] = tax['PREMISEADD'].str.replace(r'\s+WASHINGTON DC.*$', '', regex=True).str.strip()

# Expand abbreviations to match FULLADDRESS format
suffix_map = {
    r'\bST\b': 'STREET', r'\bAVE\b': 'AVENUE', r'\bPL\b': 'PLACE',
    r'\bRD\b': 'ROAD', r'\bTER\b': 'TERRACE', r'\bCT\b': 'COURT',
    r'\bCIR\b': 'CIRCLE', r'\bDR\b': 'DRIVE', r'\bLN\b': 'LANE',
    r'\bPKWY\b': 'PARKWAY', r'\bBLVD\b': 'BOULEVARD'
}

for abbrev, full in suffix_map.items():
    tax['ADDRESS_CLEAN'] = tax['ADDRESS_CLEAN'].str.replace(abbrev, full, regex=True)

# Check alignment
print("tax ADDRESS_CLEAN samples:")
print(tax['ADDRESS_CLEAN'].dropna().head(10).tolist())
print()
print("ward3 FULLADDRESS samples:")
print(ward3['FULLADDRESS'].dropna().head(10).tolist())

/var/folders/06/q_4clwgd76x_1p20fg6tlh340000gq/T/ipykernel_7065/1546437830.py:8: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  tax['ADDRESS_CLEAN'] = tax['PREMISEADD'].str.replace(r'\s+WASHINGTON DC.*$', '', regex=True).str.strip()


tax ADDRESS_CLEAN samples:
['771 WHARF STREET SW', '771 WHARF STREET SW', '771 WHARF STREET SW', '1300 E STREET NW', '1100 OAK DRIVE SE', '1300 PENNSYLVANIA AVENUE NW', '1300 PENNSYLVANIA AVENUE NW', '900 WAHLER PLACE SE', 'NW', '1100 OHIO DRIVE SW']

ward3 FULLADDRESS samples:
['2216 38TH STREET NW', '2224 38TH STREET NW', '2222 38TH STREET NW', '2218 38TH STREET NW', '2220 38TH STREET NW', '3411 33RD PLACE NW', '1746 MASSACHUSETTS AVENUE NW', '1301 S STREET NW', '3554 APPLETON STREET NW', '3548 APPLETON STREET NW']


In [18]:
# Fix the fragmentation warning and do the join
tax = tax.copy()

# Keep only useful columns + the cleaned address
tax_slim = tax[[
    'ADDRESS_CLEAN', 'NEWTOTAL', 'NEWLAND', 'NEWIMPR',
    'SALEPRICE', 'SALEDATE', 'OWNERNAME', 'NBHDNAME', 'PROPTYPE'
]].drop_duplicates(subset='ADDRESS_CLEAN').copy()

# Join on address
ward3_tax = ward3.merge(tax_slim, left_on='FULLADDRESS', right_on='ADDRESS_CLEAN', how='left')
ward3_tax.drop(columns=['ADDRESS_CLEAN'], inplace=True)

# Check results
total = len(ward3_tax)
matched = ward3_tax['NEWTOTAL'].notna().sum()
kit_matched = ward3_tax[ward3_tax['Is_Kit_House']]['NEWTOTAL'].notna().sum()

print(f"Total rows:              {total:,}")
print(f"Matched to tax data:     {matched:,} ({matched/total*100:.1f}%)")
print(f"Kit houses with values:  {kit_matched} of {ward3_tax['Is_Kit_House'].sum()}")
print()

# Quick comparison — kit vs non-kit assessed values
kit_vals = ward3_tax[ward3_tax['Is_Kit_House'] & ward3_tax['NEWTOTAL'].notna()]['NEWTOTAL']
nonkit_vals = ward3_tax[~ward3_tax['Is_Kit_House'] & ward3_tax['NEWTOTAL'].notna()]['NEWTOTAL']

print(f"Kit house avg assessed value:     ${kit_vals.mean():,.0f}")
print(f"Non-kit house avg assessed value: ${nonkit_vals.mean():,.0f}")
print(f"Kit house median assessed value:  ${kit_vals.median():,.0f}")
print(f"Non-kit house median assessed value: ${nonkit_vals.median():,.0f}")

# Save
ward3_tax.to_csv('/Users/jwitter24/Desktop/Final Project/ward3_final.csv', index=False)
print("\nSaved: ward3_final.csv updated with tax assessment data")

Total rows:              35,663
Matched to tax data:     31,983 (89.7%)
Kit houses with values:  116 of 119

Kit house avg assessed value:     $1,557,204
Non-kit house avg assessed value: $3,470,664
Kit house median assessed value:  $1,409,245
Non-kit house median assessed value: $1,396,660

Saved: ward3_final.csv updated with tax assessment data


In [19]:
import pandas as pd
import numpy as np
import folium

df = pd.read_csv('/Users/jwitter24/Desktop/Final Project/ward3_final.csv', low_memory=False)
kits = df[df['Is_Kit_House'] & df['MAR_LATITUDE'].notna()].copy()
print(f"Mapping {len(kits)} kit houses...")

def decade_color(year):
    if year < 1920: return 'blue'
    elif year < 1930: return 'red'
    elif year < 1940: return 'orange'
    else: return 'gray'

def value_to_radius(val):
    """Scale dot size by assessed value — min 5, max 18"""
    if pd.isna(val): return 5
    if val < 500000: return 5
    elif val < 1000000: return 7
    elif val < 1500000: return 9
    elif val < 2000000: return 12
    else: return 18

m = folium.Map(location=[38.93, -77.07], zoom_start=13, tiles='CartoDB positron')

#Streetcar lines again, for context. I kept the streetcar lines on the map to provide historical context for the development patterns in Ward 3. By showing the locations of these streetcar lines, we can help explain why certain areas may have more kit houses or different development patterns based on their proximity to public transit at the time. This adds depth to the analysis and helps us understand the historical factors that influenced where kit houses were built in Ward 3.
glen_echo_line = [
    [38.938583, -77.113722],[38.935667, -77.112806],[38.931889, -77.110056],
    [38.927056, -77.105333],[38.922000, -77.102611],[38.919306, -77.100333],
    [38.909639, -77.092333],[38.906222, -77.084639],[38.905917, -77.071472],
    [38.906056, -77.062861],
]
wisconsin_line = [
    [38.906056, -77.062861],[38.915694, -77.067722],[38.922917, -77.073167],
    [38.928667, -77.073111],[38.934111, -77.072361],[38.948833, -77.080111],
    [38.951083, -77.080722],[38.960778, -77.085750],
]
river_road_line = [
    [38.948833, -77.080111],[38.949028, -77.080667],[38.956472, -77.091250],
]
connecticut_ave_line = [
    [38.923556, -77.051444],[38.967556, -77.077111],
]

folium.PolyLine(glen_echo_line, color='#2196F3', weight=4, opacity=0.7, tooltip='Glen Echo Line').add_to(m)
folium.PolyLine(wisconsin_line, color='#9C27B0', weight=4, opacity=0.7, tooltip='Wisconsin Ave Line').add_to(m)
folium.PolyLine(river_road_line, color='#FF9800', weight=4, opacity=0.7, tooltip='River Road Line').add_to(m)
folium.PolyLine(connecticut_ave_line, color='#E91E63', weight=4, opacity=0.7, tooltip='Connecticut Ave Line').add_to(m)

for _, row in kits.iterrows():
    color = decade_color(row['MapYear'])
    radius = value_to_radius(row['NEWTOTAL'])
    assessed = f"${row['NEWTOTAL']:,.0f}" if pd.notna(row['NEWTOTAL']) else "N/A"
    sale = f"${row['SALEPRICE']:,.0f} ({row['SALEDATE'][:4]})" if pd.notna(row.get('SALEPRICE')) and pd.notna(row.get('SALEDATE')) else "N/A"

    folium.CircleMarker(
        location=[row['MAR_LATITUDE'], row['MAR_LONGITUDE']],
        radius=radius,
        popup=(
            f"<b>{row['FULLADDRESS']}</b><br>"
            f"Built: {row['MapYear']}<br>"
            f"Builder: {row['Builder']}<br>"
            f"Assessed Value: {assessed}<br>"
            f"Last Sale: {sale}"
        ),
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.8
    ).add_to(m)

#Legend
legend_html = """
<div style="position: fixed; bottom: 40px; left: 40px; z-index: 1000;
     background-color: white; padding: 12px; border-radius: 8px;
     border: 1px solid #ccc; font-family: Arial; font-size: 13px;">
  <b>Kit House Era</b><br>
  <span style="color:blue">●</span> Pre-1920s<br>
  <span style="color:red">●</span> 1920s<br>
  <span style="color:orange">●</span> 1930s<br>
  <span style="color:gray">●</span> 1940s+<br><br>
  <b>Dot Size = Assessed Value (2025)</b><br>
  ● &lt; $500k<br>
  ● $500k–$1M<br>
  ● $1M–$1.5M<br>
  ● $1.5M–$2M<br>
  ● &gt; $2M<br><br>
  <b>Streetcar Lines (c. 1920s)</b><br>
  <span style="color:#2196F3">━</span> Glen Echo Line<br>
  <span style="color:#9C27B0">━</span> Wisconsin Ave Line<br>
  <span style="color:#FF9800">━</span> River Road Line<br>
  <span style="color:#E91E63">━</span> Connecticut Ave Line
</div>
"""
m.get_root().html.add_child(folium.Element(legend_html))

m

Mapping 119 kit houses...


In [20]:
import folium

# Calculate values from joined dataframe
avg_kit_value = kit_vals.mean()
us_median = 366019

# Define the HTML for the top-right corner
stats_html = f'''
<div style="
    position: fixed; 
    top: 20px; right: 20px; width: 260px; height: auto; 
    background-color: white; border:2px solid #333; z-index:9999; font-size:14px;
    padding: 12px;
    border-radius: 8px;
    box-shadow: 3px 3px 6px rgba(0,0,0,0.3);
    font-family: 'Arial', sans-serif;
    ">
    <b style="font-size: 15px;">Value Comparison</b><br>
    <hr style="margin: 8px 0; border: 0; border-top: 1px solid #ccc;">
    <table style="width: 100%;">
        <tr>
            <td>Avg US Home:</td>
            <td style="text-align: right;"><b>${us_median:,.0f}</b></td>
        </tr>
        <tr>
            <td>Avg Kit House Lot Price:</td>
            <td style="text-align: right;"><b>${avg_kit_value:,.0f}</b></td>
        </tr>
    </table>
    <div style="font-size: 10px; margin-top: 8px; color: #666; line-height: 1.2;">
        <i>Note: Kit house average reflects assessed values for matched properties in Ward 3 (2025).</i>
    </div>
</div>
'''
m.get_root().html.add_child(folium.Element(stats_html))

# Display the map
m